In [29]:
import tensorflow as tf
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
from keras.models import Model
from keras.layers import Dense
from keras.applications.vgg19 import VGG19
from keras.preprocessing.image import ImageDataGenerator
from keras.callbacks import ModelCheckpoint,EarlyStopping
import pathlib

In [30]:
Batch_size = 16
Input_shape = (224,224,3)

model = VGG19(input_shape=Input_shape)

In [31]:
data_path = "dataset/training"

dataset_path = pathlib.Path(data_path)

print(len(list(dataset_path.glob("*\*.jpg"))))

5712


In [32]:
class_names = np.array([item.name for item in dataset_path.glob("*")])

In [33]:
class_names

array(['glioma', 'meningioma', 'notumor', 'pituitary'], dtype='<U10')

In [34]:
model.layers.pop()

In [35]:
for layer in model.layers[:-4]:
    layer.trainable = False

In [36]:
output = Dense(units=len(class_names),activation="softmax")

output = output(model.layers[-1].output)

model = Model(inputs =model.input,outputs=output)

In [43]:
model.compile(optimizer="adam",loss="categorical_crossentropy",metrics=['accuracy'])


In [38]:
es = EarlyStopping(monitor='val_loss',patience=5)
mc = ModelCheckpoint(filepath="vggmodel.h5",monitor="val_loss",verbose=1,save_best_only=True)


In [39]:
image_generator = ImageDataGenerator(shear_range=0.2,zoom_range=0.5,rescale=1/255.0,height_shift_range=0.2,
                                     horizontal_flip=True)

In [40]:
train_dataset = image_generator.flow_from_directory(directory=str("dataset\Training"),classes=list(class_names),batch_size=Batch_size,target_size=(224,224))

test_dataset = image_generator.flow_from_directory(directory=str("dataset\Testing"),classes=list(class_names),target_size=(224,224))

Found 5712 images belonging to 4 classes.
Found 1311 images belonging to 4 classes.


In [41]:
steps  = np.ceil(train_dataset.samples/Batch_size)

In [44]:
history = model.fit(train_dataset,epochs=20,validation_data=(test_dataset),steps_per_epoch=steps,callbacks=[es,mc])

Epoch 1/20
 60/357 [====>.........................] - ETA: 14:02 - loss: 1.3867 - accuracy: 0.2583

KeyboardInterrupt: 

In [ ]:
evaluation = model.evaluate(test_dataset)
print("Val loss:", evaluation[0])
print("Val Accuracy:", evaluation[1]*100)

acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Test'], loc='upper left')
plt.show()

plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Test'], loc='upper left')
plt.show()